In [1]:
import zarr
import dask.array as da
import pandas as pd
import numpy as np
import napari
import os
from tqdm.auto import tqdm

In [2]:
df = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/macrohet_results/dfs/sc_df.pkl')

OSError: [Errno 19] No such device: '/mnt/OPERA3/Nathan/data/macrohet/macrohet_results/dfs/sc_df.pkl'

In [ ]:
df.keys()

In [ ]:
subset_df = df[df['mtb_origin']=='Growth']
subset_df

In [26]:
n_tracks = subset_df.groupby(['Experiment ID', 'Acquisition ID']).size()

In [27]:
n_tracks.nlargest(1)

Experiment ID  Acquisition ID
PS0000         (4, 5)            3883
dtype: int64

In [28]:
image_fn = '/mnt/OPERA3/Nathan/data/macrohet/PS0000/acquisition/zarr/(4, 5).zarr'
images = da.from_zarr(f"{image_fn}/images").max(axis=2)

In [29]:
images

dask.array<max-aggregate, shape=(75, 2, 6048, 6048), dtype=uint16, chunksize=(1, 1, 6048, 6048), chunktype=numpy.ndarray>

In [30]:
# Selects rows where 'Experiment ID' equals 'PS0000' AND 'Acquisition ID' equals '(4, 5)'.
tracks = df[(df['Experiment ID'] == 'PS0000') & (df['Acquisition ID'] == (4, 5) )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
tracks

array([[1006.        ,    4.        , 1189.13110352,  926.14569092],
       [1006.        ,    5.        , 1176.62390137,  931.15734863],
       [1006.        ,    6.        , 1190.92285156,  946.45581055],
       ...,
       [ 996.        ,   72.        ,  684.67504883,  639.98370361],
       [ 996.        ,   73.        ,  688.83764648,  635.66717529],
       [ 996.        ,   74.        ,  685.81958008,  635.29003906]],
      shape=(16575, 4))

In [36]:
camera_data_tuples = [
    ('center', (39.75, 3023.5, 3023.5)),
    ('zoom', 0.139484126984127),
    ('angles', (30.987817719483903, 35.572853672744216, 228.907463346476064)),
    ('perspective', 0.0),
]

# Convert the list of (key, value) tuples directly into a dictionary.
view_dictionary = dict(camera_data_tuples)
view_dictionary

{'center': (39.75, 3023.5, 3023.5),
 'zoom': 0.139484126984127,
 'angles': (30.987817719483903, 35.572853672744216, 228.90746334647608),
 'perspective': 0.0}

In [32]:
%%time
pad_frame = np.zeros((2, 6048, 6048))

CPU times: user 0 ns, sys: 169 μs, total: 169 μs
Wall time: 190 μs


In [33]:
pad_frame

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], shape=(2, 6048, 6048))

In [35]:
napari_images = np.stack([images[frame_n], pad_frame]).compute()
v = napari.Viewer(title = 'new angle')
v.add_image(napari_images, channel_axis=1, colormap=['green','magenta'], )#scale=img_scale)
print('done')

done


In [38]:

v = napari.Viewer(title = 'new angle')
print("DEBUG: 1. Viewer initialized.") # Tracepoint 1

for frame_n in tqdm(range(len(images))):
    print(f"\nDEBUG: 2. --- Starting Frame: {frame_n} ---") # Tracepoint 2
    # --- Image Layer Management ---
    if frame_n == 0:
        img_scale = (1,1,1)
        print("DEBUG: 3. Frame 0: img_scale set to (1,1,1).") # Tracepoint 3
    else:
        img_scale=(1,1,1)
        print("DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.") # Tracepoint 4
        
        # Pop existing layers
        for i in reversed(range(len(v.layers))):
            v.layers.pop(i) 
            print(f"DEBUG: 5. Layer at index {i} popped.") # Tracepoint 5
            
    try:
        napari_images = np.stack([images[frame_n], pad_frame])
        v.add_image(napari_images, channel_axis=1, colormap=['green','magenta'], )#scale=img_scale)
        print(f"DEBUG: 6. Image layer added up to frame {frame_n}.") # Tracepoint 6
    except Exception as e:
        print(f"ERROR: Image layer failed to add at frame {frame_n}: {e}")
        continue
        
    if frame_n==0:
        print("DEBUG: 7. Frame 0: Skipping track and camera update.") # Tracepoint 7
        continue
        
    # --- Tracks Filtering and Conversion ---
    try:
        current_tracks = df[(df['Experiment ID'] == 'PS0000') 
                          & (df['Acquisition ID'] == (4, 5) ) 
                          & (df['Frame'] <= frame_n )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
        print(f"DEBUG: 8. Tracks filtered and converted to numpy. Shape: {current_tracks.shape}") # Tracepoint 8
    except Exception as e:
        print(f"ERROR: Tracks data preparation failed at frame {frame_n}: {e}")
        continue
        
    try:
        v.add_tracks(current_tracks, scale = (80, 5.04, 5.04))
        print("DEBUG: 9. Tracks layer added.") # Tracepoint 9
    except Exception as e:
        print(f"ERROR: Tracks layer failed to add at frame {frame_n}: {e}")
        
    # --- Camera and View Setup ---
    v.dims.ndisplay = 3
    print("DEBUG: 10. NDisplay set to 3.") # Tracepoint 10
    
    # Apply camera settings from the dictionary
    v.camera.center = view_dictionary['center']
    print(f"DEBUG: 11. Camera Center set to: {view_dictionary['center']}") # Tracepoint 11
    v.camera.zoom = view_dictionary['zoom']
    print(f"DEBUG: 12. Camera Zoom set to: {view_dictionary['zoom']}") # Tracepoint 12
    v.camera.angles = view_dictionary['angles']
    print(f"DEBUG: 13. Camera Angles set to: {view_dictionary['angles']}") # Tracepoint 13
    v.camera.perspective = view_dictionary['perspective']
    print(f"DEBUG: 14. Camera Perspective set to: {view_dictionary['perspective']}") # Tracepoint 14
    
    # --- Screenshot ---
    # NOTE: Corrected filename from 't{i}.png' to 't{frame_n}.png' as 'i' is the layer index.
    screenshot_path = f'/mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down/t{frame_n}.png'
    try:
        v.screenshot(path=screenshot_path)
        print(f"DEBUG: 15. Screenshot taken and saved to {screenshot_path}") # Tracepoint 15
    except Exception as e:
        print(f"ERROR: Screenshot failed at frame {frame_n}: {e}")
    


DEBUG: 1. Viewer initialized.


  0%|          | 0/75 [00:00<?, ?it/s]


DEBUG: 2. --- Starting Frame: 0 ---
DEBUG: 3. Frame 0: img_scale set to (1,1,1).
DEBUG: 6. Image layer added up to frame 0.
DEBUG: 7. Frame 0: Skipping track and camera update.

DEBUG: 2. --- Starting Frame: 1 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.
DEBUG: 6. Image layer added up to frame 1.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (390, 4)
DEBUG: 9. Tracks layer added.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down/t1.png

DEBUG: 2. --- Starting Frame: 2 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


...

DEBUG: 6. Image layer added up to frame 72.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16163, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down/t72.png

DEBUG: 2. --- Starting Frame: 73 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 73.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16373, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down/t73.png

DEBUG: 2. --- Starting Frame: 74 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 74.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16575, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down/t74.png


# 

# 

In [39]:
os.makedirs('/mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/')

In [40]:
# The DataFrame is ordered by the 'Frame' column in descending order.
# This effectively reverses the time sequence for all tracks.
tracks_reversed_time = df.sort_values(by='Frame', ascending=False)

In [41]:

v = napari.Viewer(title = 'new angle')
print("DEBUG: 1. Viewer initialized.") # Tracepoint 1

for frame_n in tqdm(range(len(images))):
    print(f"\nDEBUG: 2. --- Starting Frame: {frame_n} ---") # Tracepoint 2
    # --- Image Layer Management ---
    if frame_n == 0:
        img_scale = (1,1,1)
        print("DEBUG: 3. Frame 0: img_scale set to (1,1,1).") # Tracepoint 3
    else:
        img_scale=(1,1,1)
        print("DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.") # Tracepoint 4
        
        # Pop existing layers
        for i in reversed(range(len(v.layers))):
            v.layers.pop(i) 
            print(f"DEBUG: 5. Layer at index {i} popped.") # Tracepoint 5
            
    try:
        napari_images = np.stack([images[frame_n], pad_frame])
        v.add_image(napari_images, channel_axis=1, colormap=['green','magenta'], )#scale=img_scale)
        print(f"DEBUG: 6. Image layer added up to frame {frame_n}.") # Tracepoint 6
    except Exception as e:
        print(f"ERROR: Image layer failed to add at frame {frame_n}: {e}")
        continue
        
    if frame_n==0:
        print("DEBUG: 7. Frame 0: Skipping track and camera update.") # Tracepoint 7
        continue
        
    # --- Tracks Filtering and Conversion ---
    try:
        # The DataFrame is ordered by the 'Frame' column in descending order.
# This effectively reverses the time sequence for all tracks.
        current_tracks = df[(df['Experiment ID'] == 'PS0000') 
                          & (df['Acquisition ID'] == (4, 5) ) 
                          & (df['Frame'] <= frame_n )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
        
        print(f"DEBUG: 8. Tracks filtered and converted to numpy. Shape: {current_tracks.shape}") # Tracepoint 8
    except Exception as e:
        print(f"ERROR: Tracks data preparation failed at frame {frame_n}: {e}")
        continue
        
    try:
        v.add_tracks(current_tracks, scale = (80, 5.04, 5.04))
        print("DEBUG: 9. Tracks layer added.") # Tracepoint 9
    except Exception as e:
        print(f"ERROR: Tracks layer failed to add at frame {frame_n}: {e}")
        
    # --- Camera and View Setup ---
    v.dims.ndisplay = 3
    print("DEBUG: 10. NDisplay set to 3.") # Tracepoint 10
    
    # Apply camera settings from the dictionary
    v.camera.center = view_dictionary['center']
    print(f"DEBUG: 11. Camera Center set to: {view_dictionary['center']}") # Tracepoint 11
    v.camera.zoom = view_dictionary['zoom']
    print(f"DEBUG: 12. Camera Zoom set to: {view_dictionary['zoom']}") # Tracepoint 12
    v.camera.angles = view_dictionary['angles']
    print(f"DEBUG: 13. Camera Angles set to: {view_dictionary['angles']}") # Tracepoint 13
    v.camera.perspective = view_dictionary['perspective']
    print(f"DEBUG: 14. Camera Perspective set to: {view_dictionary['perspective']}") # Tracepoint 14
    
    # --- Screenshot ---
    # NOTE: Corrected filename from 't{i}.png' to 't{frame_n}.png' as 'i' is the layer index.
    screenshot_path = f'/mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/t{frame_n}.png'
    try:
        v.screenshot(path=screenshot_path)
        print(f"DEBUG: 15. Screenshot taken and saved to {screenshot_path}") # Tracepoint 15
    except Exception as e:
        print(f"ERROR: Screenshot failed at frame {frame_n}: {e}")
    


DEBUG: 1. Viewer initialized.


  0%|          | 0/75 [00:00<?, ?it/s]


DEBUG: 2. --- Starting Frame: 0 ---
DEBUG: 3. Frame 0: img_scale set to (1,1,1).
DEBUG: 6. Image layer added up to frame 0.
DEBUG: 7. Frame 0: Skipping track and camera update.

DEBUG: 2. --- Starting Frame: 1 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.
DEBUG: 6. Image layer added up to frame 1.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (390, 4)
DEBUG: 9. Tracks layer added.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/t1.png

DEBUG: 2. --- Starting Frame: 2 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


...

DEBUG: 6. Image layer added up to frame 72.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16163, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/t72.png

DEBUG: 2. --- Starting Frame: 73 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 73.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16373, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/t73.png

DEBUG: 2. --- Starting Frame: 74 ---
DEBUG: 4. Frame > 0: img_scale set to (1,1,1). Starting layer pop.
DEBUG: 5. Layer at index 2 popped.
DEBUG: 5. Layer at index 1 popped.
DEBUG: 5. Layer at index 0 popped.


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


DEBUG: 6. Image layer added up to frame 74.
DEBUG: 8. Tracks filtered and converted to numpy. Shape: (16575, 4)
DEBUG: 9. Tracks layer added.
DEBUG: 10. NDisplay set to 3.
DEBUG: 11. Camera Center set to: (39.75, 3023.5, 3023.5)
DEBUG: 12. Camera Zoom set to: 0.139484126984127
DEBUG: 13. Camera Angles set to: (30.987817719483903, 35.572853672744216, 228.90746334647608)
DEBUG: 14. Camera Perspective set to: 0.0
DEBUG: 15. Screenshot taken and saved to /mnt/OPERA3/Nathan/hpig_tracking_animation_upside_down_rev/t74.png
